# LeNet-5 on MNIST and CIFAR-10 with PyTorch

## 0. Introduction & Cloud Setup
This notebook demonstrates the implementation of the LeNet-5 architecture. We will cover:
1.  **Data Preparation**: Calculating dataset statistics (Mean/Std) for normalization.
2.  **Model Implementation**: LeNet-5 with explicit weight initialization.
3.  **Training**: MNIST (easier) and CIFAR-10 (harder).
4.  **Student Task**: Improve CIFAR-10 performance.

### **Running on Google Colab?**
If you are running this on Google Colab, make sure to enable the GPU:
1.  Click on `Runtime` in the top menu.
2.  Select `Change runtime type`.
3.  Under `Hardware accelerator`, select `T4 GPU` (or any available GPU).
4.  Click `Save`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time

## 1. Device Configuration

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Device: MPS (Apple Silicon Metal Performance Shaders)")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Device: CUDA ({torch.cuda.get_device_name(0)})")
    else:
        device = torch.device("cpu")
        print("Device: CPU")
    return device

device = get_device()

## 2. LeNet-5 Model Definition (with Weight Initialization)
We add a custom `_init_weights` method to properly initialize layers using Xavier (Glorot) or Kaiming (He) initialization.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10, input_channels=1):
        super(LeNet5, self).__init__()
        # Layer 1
        self.conv1 = nn.Conv2d(in_channels=input_channels, out_channels=6, kernel_size=5, stride=1)
        self.act1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        # Layer 2
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.act2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        # Layer 3: Fully Connected
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.act3 = nn.Tanh()
        
        # Layer 4: Fully Connected
        self.fc2 = nn.Linear(120, 84)
        self.act4 = nn.Tanh()
        
        # Layer 5: Output
        self.fc3 = nn.Linear(84, num_classes)

        # Explicit Weight Initialization
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                # Xavier Initialization is often good for Tanh/Sigmoid activations
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.pool1(self.act1(self.conv1(x)))
        x = self.pool2(self.act2(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5) # Flatten
        x = self.act3(self.fc1(x))
        x = self.act4(self.fc2(x))
        x = self.fc3(x)
        return x

## 3. Helper Functions

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs=10, dataset_name="Dataset"):
    print(f"\nStarting training on {dataset_name}...")
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}, Acc: {100 * correct / total:.2f}%")
                
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}] Finished. Avg Loss: {epoch_loss:.4f}, Avg Acc: {epoch_acc:.2f}%")
        
    end_time = time.time()
    print(f"Training finished in {end_time - start_time:.2f} seconds.")

def evaluate_model(model, test_loader, dataset_name="Dataset"):
    print(f"\nEvaluating on {dataset_name} Test Set...")
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Accuracy on {dataset_name} Test Images: {100 * correct / total:.2f}%")

## 4. Part 1: MNIST Dataset

### 4.1 Calculate Mean and Standard Deviation
Before training, it is good practice to normalize the data. We will calculate the mean and standard deviation of the training set.

In [ ]:
# Download Raw MNIST to calculate stats
raw_mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

# Calculate Mean and Std
print("Calculating MNIST Mean and Std...")
loader = torch.utils.data.DataLoader(raw_mnist, batch_size=10000, shuffle=False)
mean = 0.
std = 0.
total_images_count = 0

for images, _ in loader:
    batch_samples = images.size(0) 
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images_count += batch_samples

mean /= total_images_count
std /= total_images_count

print(f"MNIST Mean: {mean}")
print(f"MNIST Std: {std}")

# Expected: Mean ~0.1307, Std ~0.3081

In [ ]:
model_mnist = LeNet5(num_classes=10, input_channels=1).to(device)

In [ ]:
model_mnist

In [ ]:
model_mnist.conv1.weight.shape

In [ ]:
model_mnist.conv2.weight.shape

### 4.2 Tranining MNIST

In [ ]:
# Define Transforms with calculated (or known) values
transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset_mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)
test_dataset_mnist = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

train_loader_mnist = torch.utils.data.DataLoader(train_dataset_mnist, batch_size=64, shuffle=True)
test_loader_mnist = torch.utils.data.DataLoader(test_dataset_mnist, batch_size=64, shuffle=False)

model_mnist = LeNet5(num_classes=10, input_channels=1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_mnist = optim.Adam(model_mnist.parameters(), lr=0.001)

In [ ]:
train_dataset_mnist[0][0].shape, train_dataset_mnist[0][1]

In [ ]:
train_dataset_mnist[0][0].permute(1, 2, 0).shape

In [ ]:
idx = 355
train_dataset_mnist[idx][1]

In [ ]:
plt.imshow(train_dataset_mnist[idx][0].permute(1, 2, 0))
plt.show()

In [ ]:
train_model(model_mnist, train_loader_mnist, criterion, optimizer_mnist, num_epochs=5, dataset_name="MNIST")
evaluate_model(model_mnist, test_loader_mnist, dataset_name="MNIST")

## 5. Part 2: CIFAR-10 Dataset

### 5.1 Calculate Mean and Standard Deviation (Student Task)
Check these values against commonly cited CIFAR-10 stats: `Mean: (0.4914, 0.4822, 0.4465), Std: (0.2023, 0.1994, 0.2010)`

In [ ]:
print("Calculating CIFAR-10 Mean and Std...")
raw_cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
loader = torch.utils.data.DataLoader(raw_cifar, batch_size=10000, shuffle=False)

mean = 0.
std = 0.
total_images_count = 0

for images, _ in loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images_count += batch_samples

mean /= total_images_count
std /= total_images_count

print(f"CIFAR-10 Calculated Mean: {mean}")
print(f"CIFAR-10 Calculated Std: {std}")

### 5.2 Train Baseline Model on CIFAR-10

In [ ]:
transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset_cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar, shuffle=True)
test_dataset_cifar = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar, shuffle=False)

train_loader_cifar = torch.utils.data.DataLoader(train_dataset_cifar, batch_size=64, shuffle=True)
test_loader_cifar = torch.utils.data.DataLoader(test_dataset_cifar, batch_size=64, shuffle=False)

model_cifar = LeNet5(num_classes=10, input_channels=3).to(device)
optimizer_cifar = optim.Adam(model_cifar.parameters(), lr=0.001)

train_model(model_cifar, train_loader_cifar, criterion, optimizer_cifar, num_epochs=10, dataset_name="CIFAR-10")
evaluate_model(model_cifar, test_loader_cifar, dataset_name="CIFAR-10")

## 6. Student Task: Improve CIFAR-10 Performance

The baseline LeNet-5 model likely achieves ~50-60% accuracy on CIFAR-10. Your task is to improve this to **>70%**.

### Strategies to try:
1.  **Data Augmentation**: Add `RandomHorizontalFlip`, `RandomCrop` to `transform_cifar`.
2.  **Regularization**: Add `nn.Dropout` layer after Fully Connected layers.
3.  **Better Optimization**: Try `SGD` with Momentum and Weight Decay.
4.  **Batch Normalization**: Add `nn.BatchNorm2d` after convolutions.

### Your Code Below:

In [ ]:
class ImprovedLeNet(nn.Module):
    def __init__(self, num_classes=10, input_channels=3):
        super(ImprovedLeNet, self).__init__()
        # TODO: Define your improved architecture here
        pass

    def forward(self, x):
        # TODO: Define forward pass
        return x

# TODO: Create new transforms with augmentation
# improved_transform = ...

# TODO: Train your improved model